<a href="https://colab.research.google.com/github/li00140/ELSA-Depression-Classification/blob/main/SMOTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from collections import Counter
from pathlib import Path

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, balanced_accuracy_score
)
from sklearn.impute            import SimpleImputer

from imblearn.over_sampling    import SMOTE, SVMSMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine          import SMOTETomek, SMOTEENN
from imblearn.under_sampling   import RandomUnderSampler
from imblearn.pipeline         import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
np.random.seed(42)

In [22]:
DATA_DIR   = Path(".")
OUTPUT_DIR = Path("smote_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

WAVE_FILES = {
    1:  "wave_1_core_data_v3.csv",
    2:  "wave_2_core_data_v4.csv",
    3:  "wave_3_elsa_data_v4.csv",
    4:  "wave_4_elsa_data_v3.csv",
    5:  "wave_5_elsa_data_v4.csv",
    6:  "wave_6_elsa_data_v2.csv",
    7:  "wave_7_elsa_data.csv",
    8:  "wave_8_elsa_data_eul_v2.csv",
    9:  "wave_9_elsa_data_eul_v1.csv",
    10: "wave_10_elsa_data_eul_v1.csv",
}

CESD_POS = ["psceda", "pscedc", "pscede", "pscedg"]
CESD_NEG = ["pscedb", "pscedd", "pscedf", "pscedh"]
CESD_ALL = CESD_POS + CESD_NEG
CESD_THRESHOLD = 4

# Core features
FEATURE_MAP = {
    "age":      "indager",
    "sex":      "dhsex",
    "srhealth": "hehelf",
    "longill":  "heill",
    "limitill": "helim",
    "smoking":  "hesmk",
    "cogmem":   "cfmscr",
}
FEATURE_NAMES = list(FEATURE_MAP.keys())

MISSING_CODES = [-1, -8, -9]


In [23]:
import requests

# Helper function to resolve column names, assuming it looks for exact match or contains
def resolve_columns(all_cols, needed_cols):
    resolved = {}
    for nc in needed_cols:
        # First try exact match (case-insensitive for robustness)
        for ac in all_cols:
            if ac.lower() == nc.lower():
                resolved[nc] = ac
                break
        # If not found, try partial match (contains)
        if nc not in resolved:
            for ac in all_cols:
                if nc.lower() in ac.lower():
                    resolved[nc] = ac
                    break
    return resolved

In [24]:
def load_wave(wave_num, data_dir):
    fname = WAVE_FILES.get(wave_num)
    fpath = data_dir / fname
    if not fpath.exists():
        print(f"  [!] Wave {wave_num}: not found at {fpath}")
        return None

    all_cols  = pd.read_csv(fpath, nrows=0).columns.tolist()
    cesd_map  = resolve_columns(all_cols, CESD_ALL)
    feat_map  = resolve_columns(all_cols, list(FEATURE_MAP.values()))
    id_map    = resolve_columns(all_cols, ["idauniq"])

    needed = (list(cesd_map.values()) + list(feat_map.values()) + list(id_map.values()))

    if len(cesd_map) < 8:
        print(f"  [!] Wave {wave_num}: only {len(cesd_map)}/8 CES-D items — skipping")
        return None

    df = pd.read_csv(fpath, usecols=needed, low_memory=False)

    # Rename to canonical lower-case names
    rename = {v: k for k, v in cesd_map.items()}
    rename.update({v: k for k, v in id_map.items()})
    inv_feat = {v: k for k, v in FEATURE_MAP.items()}
    for raw_col, actual in feat_map.items():
        rename[actual] = inv_feat.get(raw_col, raw_col)
    df.rename(columns=rename, inplace=True)

    # Replace ELSA missing codes with NaN
    for col in CESD_ALL + FEATURE_NAMES:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].replace(MISSING_CODES, np.nan)

    # CES-D score
    for col in CESD_POS:
        df[f"_{col}_s"] = (df[col] == 1).astype(float)
    for col in CESD_NEG:
        df[f"_{col}_s"] = (df[col] == 2).astype(float)

    score_cols    = [f"_{c}_s" for c in CESD_ALL]
    df["cesd_score"] = df[score_cols].sum(axis=1)
    df = df[df[CESD_ALL].notna().all(axis=1)].copy()
    df["depressed"] = (df["cesd_score"] >= CESD_THRESHOLD).astype(int)
    df["wave"]      = wave_num

    keep = ["idauniq","wave","cesd_score","depressed"] + FEATURE_NAMES
    return df[[c for c in keep if c in df.columns]]


def load_all_waves(data_dir):
    frames = []
    for w in sorted(WAVE_FILES.keys()):
        print(f"  Loading wave {w}...")
        df = load_wave(w, data_dir)
        if df is not None:
            frames.append(df)
            print(f"    -> {len(df):,} valid CES-D respondents | "
                  f"depressed: {df['depressed'].sum():,} "
                  f"({df['depressed'].mean()*100:.1f}%)")
    combined = pd.concat(frames, ignore_index=True)
    print(f"\n  Total pooled: {len(combined):,} observations across {len(frames)} waves\n")
    return combined

The previous error occurred because the necessary CSV data files were not found. I will now download these files to ensure the `load_all_waves` function can access them.

In [25]:
# Base URL where the ELSA data files are hosted
BASE_URL = "https://www.elsa-project.ac.uk/sites/default/files/data/"

print("Downloading ELSA data files...")
for wave_num, filename in WAVE_FILES.items():
    file_path = DATA_DIR / filename
    if not file_path.exists():
        url = BASE_URL + filename
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
            with open(file_path, 'wb') as f:
                f.write(response.content)
            print(f"  Downloaded: {filename}")
        except requests.exceptions.RequestException as e:
            print(f"  [!] Failed to download {filename}: {e}")
    else:
        print(f"  Already exists: {filename}")
print("Download complete.")

  Already exists: wave_1_core_data_v3.csv
  Already exists: wave_2_core_data_v4.csv
  Already exists: wave_3_elsa_data_v4.csv
  Already exists: wave_4_elsa_data_v3.csv
  Already exists: wave_5_elsa_data_v4.csv
  Already exists: wave_6_elsa_data_v2.csv
  Already exists: wave_7_elsa_data.csv
  Already exists: wave_8_elsa_data_eul_v2.csv
  Already exists: wave_9_elsa_data_eul_v1.csv
  Already exists: wave_10_elsa_data_eul_v1.csv
Download complete.


In [26]:
def investigate_imbalance(df):
    print("\n" + "=" * 60)
    print("SECTION 1: CLASS IMBALANCE INVESTIGATION")
    print("=" * 60)

    fig = plt.figure(figsize=(18, 12))
    fig.suptitle("ELSA Depression (CES-D >= 4) — Class Imbalance Investigation",
                 fontsize=14, fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

    counts = df["depressed"].value_counts().sort_index()
    ir     = counts[0] / counts[1]

    # Overall bar
    ax1 = fig.add_subplot(gs[0, 0])
    bars = ax1.bar(["Not Depressed\n(CES-D < 4)", "Depressed\n(CES-D >= 4)"],
                   counts.values, color=["#4878CF","#D65F5F"],
                   edgecolor="white", linewidth=1.5)
    for bar, val in zip(bars, counts.values):
        ax1.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + counts.max()*0.02,
                 f"{val:,}\n({val/len(df)*100:.1f}%)",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax1.set_title("Overall Class Distribution", fontweight="bold")
    ax1.set_ylabel("Count")
    ax1.annotate(f"Imbalance Ratio: {ir:.2f}:1",
                 xy=(0.5, 0.92), xycoords="axes fraction", ha="center",
                 fontsize=9, color="darkred",
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFE4E4"))

    # Per-wave bar
    ax2 = fig.add_subplot(gs[0, 1])
    ws = df.groupby("wave")["depressed"].agg(total="count", dep="sum")
    ws["ndep"] = ws["total"] - ws["dep"]
    x, bw = np.arange(len(ws)), 0.35
    ax2.bar(x - bw/2, ws["ndep"], bw, label="Not Depressed", color="#4878CF", alpha=0.85)
    ax2.bar(x + bw/2, ws["dep"],  bw, label="Depressed",     color="#D65F5F", alpha=0.85)
    ax2.set_xticks(x)
    ax2.set_xticklabels([f"W{w}" for w in ws.index], fontsize=9)
    ax2.set_title("Class Distribution by Wave", fontweight="bold")
    ax2.set_ylabel("Count"); ax2.legend(fontsize=8)

    # Prevalence trend
    ax3 = fig.add_subplot(gs[0, 2])
    ws["pct"] = ws["dep"] / ws["total"] * 100
    ax3.plot(ws.index, ws["pct"], marker="o", color="#D65F5F", lw=2, ms=7)
    ax3.fill_between(ws.index, ws["pct"], alpha=0.12, color="#D65F5F")
    ax3.axhline(df["depressed"].mean()*100, color="gray", ls="--", lw=1, label="Mean")
    ax3.set_title("Depression Prevalence (%) by Wave", fontweight="bold")
    ax3.set_xlabel("Wave"); ax3.set_ylabel("Prevalence (%)")
    ax3.legend(fontsize=8); ax3.set_xticks(ws.index)

    # CES-D score histogram
    ax4 = fig.add_subplot(gs[1, 0])
    sd     = df["cesd_score"].value_counts().sort_index()
    colors = ["#D65F5F" if s >= CESD_THRESHOLD else "#4878CF" for s in sd.index]
    ax4.bar(sd.index, sd.values, color=colors, edgecolor="white", lw=0.8)
    ax4.axvline(CESD_THRESHOLD - 0.5, color="black", ls="--", lw=1.5)
    ax4.set_title("CES-D Score Distribution", fontweight="bold")
    ax4.set_xlabel("CES-D Score (0-8)"); ax4.set_ylabel("Count")
    from matplotlib.patches import Patch
    ax4.legend(handles=[
        Patch(color="#4878CF", label="Not Depressed"),
        Patch(color="#D65F5F", label="Depressed"),
        plt.Line2D([0],[0], color="black", ls="--", label="Cut-off (4)")
    ], fontsize=8)

    # By sex
    ax5 = fig.add_subplot(gs[1, 1])
    df_s = df.dropna(subset=["sex"])
    sex_dep = df_s.groupby(["sex","depressed"]).size().unstack(fill_value=0)
    sex_dep.index = ["Male","Female"]
    (sex_dep.div(sex_dep.sum(axis=1),axis=0)*100).plot(
        kind="bar", ax=ax5, color=["#4878CF","#D65F5F"],
        edgecolor="white", rot=0, legend=False)
    ax5.set_title("Class Distribution by Sex (%)", fontweight="bold")
    ax5.set_ylabel("Percentage (%)")
    ax5.legend(["Not Depressed","Depressed"], fontsize=8)
    for c in ax5.containers: ax5.bar_label(c, fmt="%.1f%%", fontsize=8)

    # By age group
    ax6 = fig.add_subplot(gs[1, 2])
    df_a = df.dropna(subset=["age"]).copy()
    df_a["age_grp"] = pd.cut(df_a["age"], bins=[49,59,69,79,120],
                              labels=["50-59","60-69","70-79","80+"])
    age_dep = df_a.groupby(["age_grp","depressed"]).size().unstack(fill_value=0)
    (age_dep.div(age_dep.sum(axis=1),axis=0)*100).plot(
        kind="bar", ax=ax6, color=["#4878CF","#D65F5F"],
        edgecolor="white", rot=0, legend=False)
    ax6.set_title("Class Distribution by Age Group (%)", fontweight="bold")
    ax6.set_ylabel("Percentage (%)")
    ax6.legend(["Not Depressed","Depressed"], fontsize=8)
    for c in ax6.containers: ax6.bar_label(c, fmt="%.1f%%", fontsize=8)

    plt.savefig(OUTPUT_DIR/"1_imbalance_investigation.png", dpi=150, bbox_inches="tight")
    plt.close()

    total, n_dep = len(df), df["depressed"].sum()
    n_not = total - n_dep
    print(f"\n  Total observations : {total:,}")
    print(f"  Not Depressed (0)  : {n_not:,}  ({n_not/total*100:.1f}%)")
    print(f"  Depressed (1)      : {n_dep:,}  ({n_dep/total*100:.1f}%)")
    print(f"  Imbalance Ratio    : {n_not/n_dep:.2f}:1")
    print(f"  -> Saved: 1_imbalance_investigation.png")

In [27]:
def prepare_xy(df):
    avail = [f for f in FEATURE_NAMES if f in df.columns]
    df_m  = df[avail + ["depressed"]].dropna()
    X = df_m[avail].values.astype(float)
    y = df_m["depressed"].values.astype(int)
    print(f"\n  Modelling dataset : {len(df_m):,} complete cases")
    print(f"  Features used     : {avail}")
    print(f"  Class balance     : {Counter(y)}")
    return X, y, avail

In [28]:
SMOTE_METHODS = {
    "No Resampling":       None,
    "Random Undersample":  RandomUnderSampler(random_state=42),
    "SMOTE":               SMOTE(random_state=42, k_neighbors=5),
    "BorderlineSMOTE":     BorderlineSMOTE(random_state=42, k_neighbors=5),
    "SVMSMOTE":            SVMSMOTE(random_state=42, k_neighbors=5),
    "ADASYN":              ADASYN(random_state=42, n_neighbors=5),
    "SMOTE+Tomek":         SMOTETomek(random_state=42),
    "SMOTE+ENN":           SMOTEENN(random_state=42),
}


def compare_smote_methods(X_train, y_train, X_test, y_test, feat_names):
    print("\n" + "=" * 60)
    print("SECTION 2: SMOTE VARIANT COMPARISON")
    print("=" * 60)

    scaler = StandardScaler()
    Xtr_sc = scaler.fit_transform(X_train)
    Xte_sc = scaler.transform(X_test)
    results = []

    for name, sampler in SMOTE_METHODS.items():
        print(f"\n  [{name}]")
        if sampler:
            Xr, yr = sampler.fit_resample(Xtr_sc, y_train)
        else:
            Xr, yr = Xtr_sc.copy(), y_train.copy()
        print(f"    Resampled dist : {Counter(yr)}")

        for clf_name, clf in [
            ("LogReg",       LogisticRegression(max_iter=1000, random_state=42)),
            ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42,
                                                     class_weight="balanced")),
        ]:
            clf.fit(Xr, yr)
            yp   = clf.predict(Xte_sc)
            prob = clf.predict_proba(Xte_sc)[:, 1]
            roc = roc_auc_score(y_test, prob)
            ap  = average_precision_score(y_test, prob)
            f1  = f1_score(y_test, yp)
            ba  = balanced_accuracy_score(y_test, yp)
            print(f"    {clf_name:15s}| ROC-AUC={roc:.3f}  F1={f1:.3f}  "
                  f"BalAcc={ba:.3f}  AP={ap:.3f}")
            results.append({
                "Resampling Method": name, "Classifier": clf_name,
                "ROC-AUC": round(roc,3), "F1 (minority)": round(f1,3),
                "Balanced Accuracy": round(ba,3), "Avg Precision": round(ap,3),
                "Majority N": Counter(yr)[0], "Minority N": Counter(yr)[1],
            })

    return pd.DataFrame(results)


def plot_smote_comparison(results_df):
    metrics = ["ROC-AUC","F1 (minority)","Balanced Accuracy","Avg Precision"]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("SMOTE Variant Comparison — ELSA Depression Classification",
                 fontsize=13, fontweight="bold")
    palette = sns.color_palette("Set2", len(SMOTE_METHODS))

    for ax, metric in zip(axes.flatten(), metrics):
        for i, clf in enumerate(results_df["Classifier"].unique()):
            sub = results_df[results_df["Classifier"] == clf].sort_values(metric)
            offset = (i - 0.5) * 0.22
            ax.barh(np.arange(len(sub)) + offset, sub[metric],
                    height=0.38, label=clf,
                    color=[palette[j] for j in range(len(sub))],
                    alpha=0.9 if i == 0 else 0.55, edgecolor="white")
        ax.set_yticks(np.arange(len(sub)))
        ax.set_yticklabels(sub["Resampling Method"], fontsize=9)
        ax.set_title(metric, fontweight="bold")
        ax.axvline(0.5, color="gray", ls="--", lw=0.8)
        if metric == metrics[0]:
            ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"2_smote_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  -> Saved: 2_smote_comparison.png")


In [29]:
def full_evaluation(X_train, y_train, X_test, y_test, feat_names):
    print("\n" + "=" * 60)
    print("SECTION 3: FULL EVALUATION — SMOTE + Random Forest")
    print("=" * 60)

    pipe = ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("smote",   SMOTE(random_state=42, k_neighbors=5)),
        ("clf",     RandomForestClassifier(n_estimators=200, random_state=42,
                                            class_weight="balanced")),
    ])

    cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_roc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_f1  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1",      n_jobs=-1)
    print(f"\n  5-Fold CV (train set):")
    print(f"    ROC-AUC : {cv_roc.mean():.3f} +/- {cv_roc.std():.3f}")
    print(f"    F1      : {cv_f1.mean():.3f}  +/- {cv_f1.std():.3f}")

    pipe.fit(X_train, y_train)
    yp   = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1]

    print("\n  Test-set Classification Report:")
    print(classification_report(y_test, yp,
          target_names=["Not Depressed","Depressed"]))

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("SMOTE + Random Forest — Final Model Evaluation",
                 fontsize=13, fontweight="bold")

    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not Dep.","Dep."],
                yticklabels=["Not Dep.","Dep."],
                ax=axes[0], cbar=False, linewidths=0.5)
    axes[0].set_title("Confusion Matrix", fontweight="bold")
    axes[0].set_ylabel("True"); axes[0].set_xlabel("Predicted")

    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    axes[1].plot(fpr, tpr, color="#D65F5F", lw=2,
                 label=f"SMOTE+RF  AUC={auc_val:.3f}")
    axes[1].plot([0,1],[0,1],"k--",lw=1)
    axes[1].fill_between(fpr, tpr, alpha=0.08, color="#D65F5F")
    axes[1].set_title("ROC Curve", fontweight="bold")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].legend(fontsize=9)

    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap_val = average_precision_score(y_test, prob)
    axes[2].plot(rec, prec, color="#4878CF", lw=2,
                 label=f"SMOTE+RF  AP={ap_val:.3f}")
    axes[2].axhline(y_test.mean(), color="gray", ls="--", lw=1,
                    label=f"Baseline ({y_test.mean():.2f})")
    axes[2].fill_between(rec, prec, alpha=0.08, color="#4878CF")
    axes[2].set_title("Precision-Recall Curve", fontweight="bold")
    axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"3_final_model_evaluation.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  -> Saved: 3_final_model_evaluation.png")

    rf = pipe.named_steps["clf"]
    feat_imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    feat_imp.plot(kind="barh", ax=ax, color="#4878CF", edgecolor="white")
    ax.set_title("Feature Importances — Random Forest (SMOTE)", fontweight="bold")
    ax.set_xlabel("Mean Decrease in Impurity")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"4_feature_importance.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  -> Saved: 4_feature_importance.png")


In [30]:
def smote_sensitivity(X_train, y_train, X_test, y_test):
    print("\n" + "=" * 60)
    print("SECTION 4: SMOTE PARAMETER SENSITIVITY")
    print("=" * 60)

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)
    results = []

    for k in [1, 3, 5, 7, 10]:
        Xr, yr = SMOTE(random_state=42, k_neighbors=k).fit_resample(Xtr, y_train)
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(Xr, yr)
        prob = clf.predict_proba(Xte)[:,1]
        results.append({"param":"k_neighbors","value":k,
                         "ROC-AUC": roc_auc_score(y_test, prob),
                         "F1":      f1_score(y_test, clf.predict(Xte))})

    for strat in [0.3, 0.5, 0.7, 0.9, 1.0]:
        Xr, yr = SMOTE(random_state=42, sampling_strategy=strat).fit_resample(Xtr, y_train)
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(Xr, yr)
        prob = clf.predict_proba(Xte)[:,1]
        results.append({"param":"sampling_strategy","value":strat,
                         "ROC-AUC": roc_auc_score(y_test, prob),
                         "F1":      f1_score(y_test, clf.predict(Xte))})

    res_df = pd.DataFrame(results)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("SMOTE Hyperparameter Sensitivity", fontsize=13, fontweight="bold")

    for ax, param in zip(axes, ["k_neighbors","sampling_strategy"]):
        sub = res_df[res_df["param"] == param]
        ax.plot(sub["value"], sub["ROC-AUC"], marker="o", color="#D65F5F",
                lw=2, label="ROC-AUC")
        ax.plot(sub["value"], sub["F1"], marker="s", color="#4878CF",
                lw=2, label="F1 (minority)")
        ax.set_title(f"Effect of {param}", fontweight="bold")
        ax.set_xlabel(param); ax.set_ylabel("Score")
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"5_smote_sensitivity.png", dpi=150, bbox_inches="tight")
    plt.close()

    for param in ["k_neighbors","sampling_strategy"]:
        sub = res_df[res_df["param"] == param]
        print(f"\n  {param}:")
        print(sub[["value","ROC-AUC","F1"]].round(3).to_string(index=False))
    print(f"\n  -> Saved: 5_smote_sensitivity.png")


In [31]:
def wave_level_analysis(df):
    print("\n" + "=" * 60)
    print("SECTION 5: PER-WAVE SMOTE ANALYSIS")
    print("=" * 60)

    records = []
    for wn in sorted(df["wave"].unique()):
        avail = [f for f in FEATURE_NAMES if f in df.columns]
        sub   = df[df["wave"] == wn][avail + ["depressed"]].dropna()
        if len(sub) < 200 or sub["depressed"].nunique() < 2:
            continue
        X = sub[avail].values; y = sub["depressed"].values
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42)

        pipe = ImbPipeline([
            ("scaler", StandardScaler()),
            ("smote",  SMOTE(random_state=42, k_neighbors=5)),
            ("clf",    RandomForestClassifier(n_estimators=100, random_state=42)),
        ])
        pipe.fit(Xtr, ytr)
        prob = pipe.predict_proba(Xte)[:,1]
        yp   = pipe.predict(Xte)

        records.append({
            "Wave": wn, "N": len(sub),
            "Dep %": round(y.mean()*100, 1),
            "ROC-AUC": round(roc_auc_score(yte, prob), 3),
            "F1":      round(f1_score(yte, yp), 3),
            "Bal. Acc": round(balanced_accuracy_score(yte, yp), 3),
        })

    wr = pd.DataFrame(records)
    print(wr.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Per-Wave SMOTE + RF Performance", fontsize=13, fontweight="bold")

    axes[0].plot(wr["Wave"], wr["ROC-AUC"],  marker="o", color="#D65F5F",
                 lw=2, label="ROC-AUC")
    axes[0].plot(wr["Wave"], wr["Bal. Acc"], marker="s", color="#4878CF",
                 lw=2, label="Balanced Accuracy")
    axes[0].set_title("Model Performance by Wave", fontweight="bold")
    axes[0].set_xlabel("Wave"); axes[0].set_ylabel("Score")
    axes[0].legend(); axes[0].set_xticks(wr["Wave"])

    axes[1].bar(wr["Wave"], wr["Dep %"], color="#D65F5F", edgecolor="white")
    axes[1].set_title("Depression Prevalence (%) by Wave", fontweight="bold")
    axes[1].set_xlabel("Wave"); axes[1].set_ylabel("Prevalence (%)")
    axes[1].set_xticks(wr["Wave"])

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"6_per_wave_analysis.png", dpi=150, bbox_inches="tight")
    plt.close()
    wr.to_csv(OUTPUT_DIR/"wave_smote_results.csv", index=False)
    print(f"\n  -> Saved: 6_per_wave_analysis.png  |  wave_smote_results.csv")


In [32]:
def save_resampled(X_train, y_train, feat_names):
    scaler = StandardScaler()
    Xr, yr = SMOTE(random_state=42, k_neighbors=5).fit_resample(
                 scaler.fit_transform(X_train), y_train)
    df_out = pd.DataFrame(Xr, columns=feat_names)
    df_out["depressed"] = yr
    out_path = OUTPUT_DIR/"elsa_smote_resampled_train.csv"
    df_out.to_csv(out_path, index=False)
    print(f"\n  Resampled training set -> {out_path}")
    print(f"  Shape: {df_out.shape}  |  Class dist: {Counter(yr)}")

In [33]:
def main():
    print("\n" + "=" * 60)
    print("ELSA SMOTE INVESTIGATION & IMPLEMENTATION")
    print("=" * 60)
    print(f"CSV folder : {DATA_DIR.resolve()}")
    print(f"Output dir : {OUTPUT_DIR.resolve()}\n")

    df = load_all_waves(DATA_DIR)
    investigate_imbalance(df)

    X, y, feat_names = prepare_xy(df)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42)
    print(f"\n  Train: {len(y_train):,}  |  Test: {len(y_test):,}")

    results_df = compare_smote_methods(X_train, y_train, X_test, y_test, feat_names)
    plot_smote_comparison(results_df)
    results_df.to_csv(OUTPUT_DIR/"smote_comparison_results.csv", index=False)
    print(f"  -> Saved: smote_comparison_results.csv")

    full_evaluation(X_train, y_train, X_test, y_test, feat_names)
    smote_sensitivity(X_train, y_train, X_test, y_test)
    wave_level_analysis(df)
    save_resampled(X_train, y_train, feat_names)

    print("\n" + "=" * 60)
    print("ALL OUTPUTS SAVED TO:", OUTPUT_DIR.resolve())
    print("=" * 60)
    print("\nFiles generated:")
    for f in sorted(OUTPUT_DIR.iterdir()):
        print(f"  {f.name:<48} {f.stat().st_size/1024:6.1f} KB")


if __name__ == "__main__":
    main()

def wave_level_analysis(df):
    print("\n" + "=" * 60)
    print("SECTION 5: PER-WAVE SMOTE ANALYSIS")
    print("=" * 60)

    records = []
    for wn in sorted(df["wave"].unique()):
        avail = [f for f in FEATURE_NAMES if f in df.columns]
        sub   = df[df["wave"] == wn][avail + ["depressed"]].dropna()
        # Modified condition: Ensure at least two unique classes and enough samples for stratified split
        if sub["depressed"].nunique() < 2 or len(sub) < 4: # Changed from < 200 to < 4
            print(f"  [!] Skipping wave {wn}: Not enough data or only one class (N={len(sub)}, Classes={sub['depressed'].nunique()})")
            continue
        X = sub[avail].values; y = sub["depressed"].values
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42)

        pipe = ImbPipeline([
            ("scaler", StandardScaler()),
            ("smote",  SMOTE(random_state=42, k_neighbors=5)),
            ("clf",    RandomForestClassifier(n_estimators=100, random_state=42)),
        ])
        pipe.fit(Xtr, ytr)
        prob = pipe.predict_proba(Xte)[:,1]
        yp   = pipe.predict(Xte)

        records.append({
            "Wave": wn, "N": len(sub),
            "Dep %": round(y.mean()*100, 1),
            "ROC-AUC": round(roc_auc_score(yte, prob), 3),
            "F1":      round(f1_score(yte, yp), 3),
            "Bal. Acc": round(balanced_accuracy_score(yte, yp), 3),
        })

    if not records:
        print("  No waves met the criteria for per-wave analysis.")
        return # Exit if no records were generated

    wr = pd.DataFrame(records)
    print(wr.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Per-Wave SMOTE + RF Performance", fontsize=13, fontweight="bold")

    axes[0].plot(wr["Wave"], wr["ROC-AUC"],  marker="o", color="#D65F5F",
                 lw=2, label="ROC-AUC")
    axes[0].plot(wr["Wave"], wr["Bal. Acc"], marker="s", color="#4878CF",
                 lw=2, label="Balanced Accuracy")
    axes[0].set_title("Model Performance by Wave", fontweight="bold")
    axes[0].set_xlabel("Wave"); axes[0].set_ylabel("Score")
    axes[0].legend(); axes[0].set_xticks(wr["Wave"])

    axes[1].bar(wr["Wave"], wr["Dep %"], color="#D65F5F", edgecolor="white")
    axes[1].set_title("Depression Prevalence (%) by Wave", fontweight="bold")
    axes[1].set_xlabel("Wave"); axes[1].set_ylabel("Prevalence (%)")
    axes[1].set_xticks(wr["Wave"])

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"6_per_wave_analysis.png", dpi=150, bbox_inches="tight")
    plt.close()
    wr.to_csv(OUTPUT_DIR/"wave_smote_results.csv", index=False)
    print(f"\n  -> Saved: 6_per_wave_analysis.png  |  wave_smote_results.csv")


ELSA SMOTE INVESTIGATION & IMPLEMENTATION
CSV folder : /content
Output dir : /content/smote_outputs

  Loading wave 1...
    -> 11,612 valid CES-D respondents | depressed: 2,120 (18.3%)
  Loading wave 2...
    -> 5,794 valid CES-D respondents | depressed: 1,068 (18.4%)
  Loading wave 3...
    -> 5,882 valid CES-D respondents | depressed: 1,041 (17.7%)
  Loading wave 4...
    -> 4,604 valid CES-D respondents | depressed: 767 (16.7%)
  Loading wave 5...
    -> 6,481 valid CES-D respondents | depressed: 1,168 (18.0%)
  Loading wave 6...
    -> 5,477 valid CES-D respondents | depressed: 821 (15.0%)
  Loading wave 7...
    -> 5,452 valid CES-D respondents | depressed: 806 (14.8%)
  Loading wave 8...
    -> 5,873 valid CES-D respondents | depressed: 898 (15.3%)
  Loading wave 9...
    -> 5,984 valid CES-D respondents | depressed: 879 (14.7%)
  Loading wave 10...
    -> 5,587 valid CES-D respondents | depressed: 978 (17.5%)

  Total pooled: 62,746 observations across 10 waves


SECTION 1: CL